# English Laya: LoRA workflow check on a Colab CPU

Reuse the English Laya checkpoint, content safety question, and Aegis revision from the earlier notebooks. A CPU can run LoRA forward passes, backpropagation, and parameter updates. Speed depends on the CPU and memory Colab assigns.

This is a **workflow check**: load the model, attach LoRA, run three updates, check gradients and weights, then save and reload.

- Select 8 training candidates and 4 separate validation examples with short, complete inputs.
- Run one example per forward pass and accumulate two examples per update, for three updates by default.
- Use the full English Laya model in FP32 on CPU, without mixed precision or quantization.
- Train the encoder LoRA and the existing decision head.
- Skip calibration and the final test set. Use this run to verify the workflow, not generalization.

**Execution status:** Model code and training have not been run locally. Run the cells in order in Colab.

In Colab, choose **Runtime → Change runtime type → Hardware accelerator: None**. Download any results you want to keep before switching runtimes; temporary files may be lost.

The GPU notebook requires a GPU to prevent accidentally starting a slow CPU run. LoRA itself can train on a CPU.

## 1. Install and check Colab dependencies

Select a Colab CPU runtime for this debug notebook. Run the installation cell in a dedicated session for this experiment.

When the cell prints **Setup passed**, choose **Runtime → Restart session** (**代码执行程序 → 重启会话**) from Colab's top menu and confirm. Wait for Colab to reconnect, then click **▶** beside the Step 2 code cell. Skip Step 1 after restarting.

Restarting clears Python's in-memory variables so Step 2 can load the newly installed packages.

The experiment pins `laya==0.3.22`, `transformers==4.57.6`, `huggingface-hub==0.36.0`, and `peft==0.18.0` to keep the model and training APIs consistent.

`transformers==4.57.6` requires `huggingface-hub>=0.34.0,<1.0`. The newer versions of `diffusers`, `gradio`, and `gradio-client` that caused the conflict require Hub 1.x. The cell removes those unused packages before installing dependencies and pins the installed PyTorch version to preserve Colab's CUDA build.

The cell installs the dependencies for weights, tables, and plots. No separate pip upgrade or Unsloth installation is needed.

The cell checks installed versions, required dependencies, and imports in a fresh Python process. A failed check stops the cell.

Version constraints: [Transformers 4.57.6](https://pypi.org/pypi/transformers/4.57.6/json), [Laya 0.3.22](https://pypi.org/pypi/laya/0.3.22/json), and [PEFT 0.18.0](https://pypi.org/pypi/peft/0.18.0/json).


In [ ]:
import os
import sys
import json
import subprocess
import tempfile
from pathlib import Path
import importlib.metadata as package_metadata

# Limit installation to a dedicated Colab session so it does not modify a local environment.
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Run the installation cell in a dedicated Colab session.") from exc
if sys.version_info < (3, 10):
    raise RuntimeError("The pinned Laya and PEFT versions require Python >= 3.10. Choose a current Colab runtime.")

WITH_PEFT = True
WITH_PARQUET = False
REQUIRES_GPU = False
NEEDS_GRAD_SCALER = False
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

try:
    colab_torch_version = package_metadata.version("torch")
except package_metadata.PackageNotFoundError as exc:
    raise RuntimeError("Colab's preinstalled PyTorch was not found. Reconnect to a standard Colab runtime.") from exc
expected_versions = {
    "laya": "0.3.22", "transformers": "4.57.6", "huggingface-hub": "0.36.0",
}
if WITH_PEFT:
    expected_versions["peft"] = "0.18.0"
extra_packages = ["safetensors", "pandas", "matplotlib", "tqdm"]
if WITH_PARQUET:
    extra_packages.append("pyarrow")

# These packages conflict with Hub 0.x and are unused by these notebooks.
subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "diffusers", "gradio", "gradio-client",
])
# Pin the installed PyTorch version to preserve Colab's CUDA build.
with tempfile.TemporaryDirectory(prefix="laya-colab-setup-") as setup_directory:
    constraint_file = Path(setup_directory) / "keep-colab-torch.txt"
    constraint_file.write_text(f"torch=={colab_torch_version}\n", encoding="utf-8")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--constraint", str(constraint_file),
        *[f"{name}=={version}" for name, version in expected_versions.items()],
        *extra_packages,
    ])

probe_config = {
    "pins": expected_versions, "torch_version": colab_torch_version,
    "roots": list(expected_versions) + ["torch"] + extra_packages,
    "with_peft": WITH_PEFT, "with_parquet": WITH_PARQUET,
    "requires_gpu": REQUIRES_GPU, "needs_grad_scaler": NEEDS_GRAD_SCALER,
}
probe_source = r"""
import importlib.metadata as md
import inspect
import json
import sys
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from packaging.version import Version

config = json.loads(sys.argv[1])
for name, expected in config["pins"].items():
    actual = md.version(name)
    if actual != expected:
        raise RuntimeError(f"{name}: installed {actual}, expected {expected}")
if md.version("torch") != config["torch_version"]:
    raise RuntimeError("PyTorch changed during setup; use the original Colab PyTorch build.")

# Check the dependencies used by this notebook, including transitive dependencies.
# Unused optional extras are not enabled here.
pending = list(config["roots"])
visited, problems = set(), []
while pending:
    name = canonicalize_name(pending.pop())
    if name in visited:
        continue
    visited.add(name)
    try:
        distribution = md.distribution(name)
    except md.PackageNotFoundError:
        problems.append(f"Missing package: {name}")
        continue
    for text in distribution.requires or []:
        requirement = Requirement(text)
        if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            actual = md.version(requirement.name)
        except md.PackageNotFoundError:
            problems.append(f"{name} requires {requirement}, but it is missing")
            continue
        if requirement.specifier and not requirement.specifier.contains(Version(actual), prereleases=True):
            problems.append(f"{name} requires {requirement}; installed {actual}")
        pending.append(requirement.name)
if problems:
    raise RuntimeError("Notebook dependency check failed:\n" + "\n".join(sorted(set(problems))))

import torch
import numpy
import pandas
import matplotlib.pyplot
import huggingface_hub
import laya
from laya.common import build_sequence, collate_items, QTYPES, render_options
from transformers import ModernBertModel
from safetensors.torch import load_file, save_file

assert callable(laya.load)
assert QTYPES["noul"] == 2
assert {"return_stats", "return_truncation_stats"} <= set(inspect.signature(build_sequence).parameters)
if config["with_peft"]:
    from peft import LoraConfig, PeftModel, get_peft_model
    LoraConfig(r=8, lora_alpha=16, target_modules="all-linear")
if config["with_parquet"]:
    import pyarrow
    import pyarrow.parquet
if config["needs_grad_scaler"] and not hasattr(torch.amp, "GradScaler"):
    raise RuntimeError("This notebook requires torch.amp.GradScaler; select a current Colab runtime.")
if config["requires_gpu"] and not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU. Switch the Colab runtime type to GPU and try again.")
"""
subprocess.check_call([sys.executable, "-c", probe_source, json.dumps(probe_config)])

# Prevent Run all from continuing with old dependencies cached in this kernel.
_LAYA_NOTEBOOK_INSTALL_PID = os.getpid()
print("Setup passed. In Colab's top menu, choose Runtime → Restart session (代码执行程序 → 重启会话).")
print("Confirm, wait for Colab to reconnect, then click ▶ beside Step 2. Skip Step 1.")


## 2. Set up the CPU experiment

With `RUN_TRAINING=False`, only the baseline runs. Set it to True when ready and rerun from this cell.

`MAX_UPDATES=3` counts parameter updates. Two examples per update give six training examples by default. The CPU still runs the full encoder even when only its LoRA parameters are updated.

Keep the original 512-token budget and select complete examples whose encoded length is at most 192. This reduces CPU work without truncating sentences.

Limit execution to two threads by default to reduce contention on small runtimes. Memory must hold the full 421M-parameter model, the decision head's optimizer state, and activations in addition to the adapter.

In [ ]:
# Restart the kernel after installation to clear previously imported dependencies.
import os as _laya_setup_os
if globals().get("_LAYA_NOTEBOOK_INSTALL_PID") == _laya_setup_os.getpid():
    raise RuntimeError(
        "In Colab's top menu, choose Runtime → Restart session (代码执行程序 → 重启会话) and confirm. "
        "Wait for Colab to reconnect, then rerun Step 2. Skip Step 1."
    )

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import copy, gc, hashlib, json, math, random, time, uuid, zipfile
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import laya
from laya.common import build_sequence, collate_items, QTYPES
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model, PeftModel
from safetensors.torch import save_file, load_file
from IPython.display import display

RUN_TRAINING = False  # Set to True to update parameters on the CPU.
DEVICE = "cpu"
SEED = 42
TRAIN_N, VALID_N = 8, 4  # Use even counts to sample equal numbers of safe and unsafe examples.
MAX_UPDATES = 3
GRAD_ACCUM = 2
MAX_LEN, HEAD_MAX_LEN = 512, 192
MAX_DEBUG_SEQUENCE = 192
LORA_R, LORA_ALPHA = 8, 16
LORA_LR, HEAD_LR = 2e-4, 1e-4
THREADS = max(1, min(2, os.cpu_count() or 1))
torch.set_num_threads(THREADS)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "convaiinnovations/laya"
MODEL_REVISION = "85cc74ddae6109077ebe2853a9b3077fb6606a78"
DATASET_ID = "nvidia/Aegis-AI-Content-Safety-Dataset-2.0"
DATASET_REVISION = "d86bb8bedff51d25ac834ab7838f1cc61acb7a2c"
QUESTIONS = {"unsafe_content": {"type": "noul", "instructions": "Does the current input contain unsafe content?"}}
QUESTION = {"t": "noul", "ins": QUESTIONS["unsafe_content"]["instructions"], "crit": None}
VERSIONS = {p: metadata.version(p) for p in ["laya", "torch", "transformers", "peft", "huggingface_hub", "safetensors"]}
OUT = Path("laya_cpu_debug") / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6])
OUT.mkdir(parents=True, exist_ok=False)
print("device:", DEVICE, "threads:", THREADS, "training:", RUN_TRAINING)
print("output:", OUT)
print("versions:", VERSIONS)

## 3. Load the full English model

Use FP32 on CPU. The first run downloads weights. Disable compilation and accelerated kernels to exercise the standard execution path.

Load the original English Laya checkpoint used in the other experiments.

In [ ]:
agent = laya.load(MODEL_ID, revision=MODEL_REVISION, device="cpu", fast=False, compile=False)
model = agent.model.float().eval()
tokenizer = agent.tok
BASE_CONFIG = copy.deepcopy(agent.cfg)
SHIPPED_T = float(agent.temperature_by_options.get("noul:2", agent.temperature[QTYPES["noul"]]))
assert next(model.parameters()).device.type == "cpu"
print("Total model parameters:", sum(p.numel() for p in model.parameters()))
print("Original noul temperature:", SHIPPED_T)

## 4. Select a small sample of real data

Read the official training and validation splits. Remove empty or REDACTED inputs, conflicts for the same ID or normalized text, and duplicates. Exclude training records that overlap the entire cleaned validation pool by ID or text.

Balance safe and unsafe labels and prefer short English inputs for this debug run. This differs from business traffic. Deduplication handles normalized exact matches only. The final test file is not downloaded.

If too few samples qualify, the notebook stops. Missing labels are never treated as safe.

In [ ]:
def clean_split(name):
    path = hf_hub_download(DATASET_ID, filename=f"{name}.json", repo_type="dataset",
                           revision=DATASET_REVISION, token=False)
    raw = pd.read_json(path)
    frame = raw.loc[raw["prompt_label"].isin(["safe", "unsafe"]), ["id", "prompt", "prompt_label"]].copy()
    frame["id"] = frame["id"].astype(str)
    frame["prompt"] = frame["prompt"].fillna("").astype(str).str.strip()
    frame = frame[frame["prompt"].ne("") & frame["prompt"].str.upper().ne("REDACTED")]
    normalized = frame["prompt"].str.casefold().str.replace(r"\s+", " ", regex=True)
    frame["text_sha256"] = normalized.map(lambda t: hashlib.sha256(t.encode()).hexdigest())
    for key in ["id", "text_sha256"]:
        counts = frame.groupby(key)["prompt_label"].nunique()
        frame = frame[~frame[key].isin(counts[counts > 1].index)].drop_duplicates(key)
    frame["gold"] = frame["prompt_label"].map({"safe": 0, "unsafe": 1}).astype(int)
    return frame.reset_index(drop=True)

valid_pool = clean_split("validation")
train_pool = clean_split("train")
train_pool = train_pool[~train_pool["id"].isin(valid_pool["id"]) &
                        ~train_pool["text_sha256"].isin(valid_pool["text_sha256"])]

def pick_short(frame, n):
    assert n >= 2 and n % 2 == 0
    selected, counts = [], {0: 0, 1: 0}
    for row in frame.sample(frac=1, random_state=SEED).to_dict("records"):
        y = row["gold"]
        if counts[y] >= n // 2 or not laya.is_english(row["prompt"]):
            continue
        ids, markers, stats = build_sequence(tokenizer, row["prompt"], QUESTION,
            max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN, return_truncation_stats=True)
        if stats["truncated"] or len(ids) > MAX_DEBUG_SEQUENCE:
            continue
        assert len(markers) == 2
        selected.append({**row, "ids": ids, "markers": markers, "qtype": QTYPES["noul"], "label": y})
        counts[y] += 1
        if len(selected) == n:
            break
    if len(selected) != n:
        raise ValueError("Too few samples meet the length and label requirements. Try increasing MAX_DEBUG_SEQUENCE slightly.")
    return selected

train_rows, valid_rows = pick_short(train_pool, TRAIN_N), pick_short(valid_pool, VALID_N)
for key in ["id", "text_sha256"]:
    assert set(r[key] for r in train_rows).isdisjoint(r[key] for r in valid_rows)
display(pd.DataFrame([{"split": split, "label": row["prompt_label"], "tokens": len(row["ids"]), "text": row["prompt"]}
                      for split, rows in [("train", train_rows), ("validation", valid_rows)] for row in rows]))
del train_pool, valid_pool
gc.collect()

## 5. Measure the debug baseline

Evaluate training candidates and the separate validation examples. `raw_CE` uses unscaled logits, matching the training objective.

`p_unsafe` uses the original model's temperature so scores can be compared before and after training. The sample is too small to fit a new temperature.

With four validation examples, one decision changes accuracy by 25 percentage points. Record accuracy as a debug result.

In [ ]:
INPUT_KEYS = ["input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype"]
def batch_for(row):
    packed = collate_items([[row]], tokenizer.pad_token_id)
    return {key: packed[key] for key in INPUT_KEYS}, packed["label"]

@torch.no_grad()
def evaluate_rows(net, rows):
    net.eval()
    logits = []
    started = time.perf_counter()
    for row in rows:
        batch, _ = batch_for(row)
        scores, _ = net(**batch)
        if not torch.isfinite(scores).all():
            raise RuntimeError("Prediction scores contain NaN or infinity.")
        logits.append(scores.detach().float().cpu())
    z = torch.cat(logits)
    y = torch.tensor([r["label"] for r in rows])
    report = {"n": len(rows), "raw_CE": F.cross_entropy(z, y).item(),
              "accuracy": (z.argmax(-1) == y).float().mean().item(),
              "seconds": time.perf_counter() - started}
    return report, torch.softmax(z / SHIPPED_T, -1)[:, 1].numpy()

baseline_train, _ = evaluate_rows(model, train_rows)
baseline_valid, baseline_p = evaluate_rows(model, valid_rows)
display(pd.DataFrame([{"split": "train", **baseline_train}, {"split": "validation", **baseline_valid}]))

## 6. Attach LoRA and verify the trainable parameters

Use the GPU experiment's rank 8, alpha 16, and all-linear targets. Train encoder LoRA and the existing head, type embedding, and scorer. Keep `act_head` frozen.

Run in FP32 without CUDA GradScaler or autocast. Disable gradient checkpointing for these short inputs to avoid recomputation on the CPU.

Save the initial LoRA values and sample a small slice of frozen weights for comparison. This check does not cover every encoder weight.

In [ ]:
if RUN_TRAINING:
    if hasattr(model.encoder, "peft_config"):
        raise RuntimeError("LoRA is already attached. Start again from Step 2 to repeat the experiment.")
    model.requires_grad_(False)
    model.encoder = get_peft_model(model.encoder, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules="all-linear", bias="none"))
    model.encoder.gradient_checkpointing_disable()
    model.head_checkpointing = False
    for module in [model.head, model.type_emb, model.scorer]:
        if module is not None:
            module.requires_grad_(True)
    for name, parameter in model.encoder.named_parameters():
        if parameter.requires_grad and "lora_" not in name:
            raise RuntimeError(f"Unexpected unfrozen encoder parameter: {name}")
    lora_initial = {name: p.detach().clone() for name, p in model.encoder.named_parameters() if p.requires_grad}
    if not lora_initial:
        raise RuntimeError("No trainable LoRA parameters found.")
    frozen_name = next(name for name, p in model.encoder.named_parameters() if not p.requires_grad)
    frozen_probe = dict(model.encoder.named_parameters())[frozen_name].detach().flatten()[:256].clone()
    print("Trainable LoRA parameters:", sum(p.numel() for n, p in model.encoder.named_parameters() if p.requires_grad))
    print("Trainable decision head parameters:", sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")))
else:
    print("Training is disabled. Set RUN_TRAINING=True in Step 2 and run the cells from there.")

## 7. Run three CPU updates

For each update, run two forward and backward passes with batch size 1, then average the accumulated gradients by sample count.

Record:

- `train_CE`: Mean loss on the two examples used for this update.
- `lora_grad_L1`: Sum of absolute LoRA gradients, to check whether gradients reach the adapter.
- `seconds`: Wall-clock time for the update, to help decide whether to scale up.

Common LoRA initializations can give some matrices zero gradient on the first backward pass. Check that the adapter receives gradients overall and that its weights change after an update.

Examples can differ between steps, so loss need not decrease on every update.

In [ ]:
def run_cpu_updates(net, rows):
    lora_params = [p for n, p in net.named_parameters() if n.startswith("encoder.") and p.requires_grad]
    head_params = [p for n, p in net.named_parameters() if not n.startswith("encoder.") and p.requires_grad]
    params = lora_params + head_params
    optimizer = torch.optim.AdamW([
        {"params": lora_params, "lr": LORA_LR}, {"params": head_params, "lr": HEAD_LR}
    ], weight_decay=0.01)
    order = list(range(len(rows)))
    random.Random(SEED).shuffle(order)
    history = []
    net.train()
    for step in range(MAX_UPDATES):
        started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        used = []
        for micro in range(GRAD_ACCUM):
            row = rows[order[(step * GRAD_ACCUM + micro) % len(order)]]
            batch, y = batch_for(row)
            scores, _ = net(**batch)
            loss = F.cross_entropy(scores.float(), y)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite CPU training loss. Stopping.")
            (loss / GRAD_ACCUM).backward()
            total += float(loss.detach())
            used.append(row["id"])
        if not all(p.grad is None or torch.isfinite(p.grad).all() for p in params):
            raise RuntimeError("Gradients contain NaN or infinity. Stopping.")
        grad_l1 = sum(float(p.grad.detach().abs().sum()) for p in lora_params if p.grad is not None)
        torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)
        optimizer.step()
        record = {"update": step + 1, "train_CE": total / GRAD_ACCUM,
                  "lora_grad_L1": grad_l1, "seconds": time.perf_counter() - started,
                  "sample_ids": used}
        history.append(record)
        print(record, flush=True)
    optimizer.zero_grad(set_to_none=True)
    return history

TRAINING_DONE = False
if RUN_TRAINING:
    history = run_cpu_updates(model, train_rows)
    changed = max(float((p.detach() - lora_initial[name]).abs().max())
                  for name, p in model.encoder.named_parameters() if name in lora_initial)
    frozen_after = dict(model.encoder.named_parameters())[frozen_name].detach().flatten()[:256]
    checks = {
        "updates_completed": len(history),
        "lora_received_nonzero_gradient": any(r["lora_grad_L1"] > 0 for r in history),
        "lora_max_weight_change": changed,
        "frozen_weight_probe_unchanged": torch.equal(frozen_probe, frozen_after),
    }
    print(checks)
    if not checks["lora_received_nonzero_gradient"] or changed == 0 or not checks["frozen_weight_probe_unchanged"]:
        raise RuntimeError("The LoRA update check failed. Inspect the gradients and parameter configuration.")
    TRAINING_DONE = True
    pd.DataFrame(history).to_csv(OUT / "update_history.csv", index=False)
    del lora_initial, frozen_probe, frozen_after
    gc.collect()
else:
    print("No parameter updates were run.")

## 8. Compare results before and after updates

Rerun the same training and validation examples with the same temperature and input format so the comparison isolates parameter updates.

Any change in loss or accuracy describes these few examples. Three updates and four validation rows cannot establish that LoRA generalizes better.

Passing the gradient and weight checks confirms that the workflow updated parameters, even if accuracy did not improve.

In [ ]:
if TRAINING_DONE:
    after_train, _ = evaluate_rows(model, train_rows)
    after_valid, after_p = evaluate_rows(model, valid_rows)
    comparison = pd.DataFrame([
        {"model": "base", "split": "train", **baseline_train},
        {"model": "lora+head", "split": "train", **after_train},
        {"model": "base", "split": "validation", **baseline_valid},
        {"model": "lora+head", "split": "validation", **after_valid},
    ])
    display(comparison)
    details = pd.DataFrame([{"text": r["prompt"], "label": r["prompt_label"],
                             "before_p_unsafe": float(b), "after_p_unsafe": float(a)}
                            for r, b, a in zip(valid_rows, baseline_p, after_p)])
    display(details)
    comparison.to_csv(OUT / "comparison.csv", index=False)
    details.to_csv(OUT / "validation_predictions.csv", index=False)
    plt.plot([h["update"] for h in history], [h["train_CE"] for h in history], marker="o")
    plt.xlabel("Optimizer update")
    plt.ylabel("Training CE (different mini-batches)")
    plt.show()
else:
    print("Complete training in Step 7 first.")

## 9. Save the adapter and decision head and verify CPU reload

Save both the LoRA adapter and the updated head. Release the model and restore both parts on the exact same base checkpoint.

Compare probabilities on two validation examples before saving and after reload. The probe passes when the maximum difference is within `1e-5`. This tolerance is for the same CPU FP32 path on these examples.

The archive contains adapter weights rather than a full merged model loadable with `laya.load(directory)`. It does not include optimizer or random state for exact resumption. Run this cell once; start again at Step 2 for a new run.

In [ ]:
if TRAINING_DONE:
    model.eval()
    probe = valid_rows[:2]
    _, expected = evaluate_rows(model, probe)
    model.encoder.save_pretrained(OUT / "encoder_adapter", safe_serialization=True, save_embedding_layers=False)
    head_state = {k: v.detach().cpu().contiguous().clone() for k, v in model.state_dict().items() if not k.startswith("encoder.")}
    if not all(torch.isfinite(v).all() for v in head_state.values()):
        raise RuntimeError("Invalid decision head weights. Refusing to save.")
    save_file(head_state, str(OUT / "decision_head.safetensors"))
    manifest = {
        "base_model": MODEL_ID, "base_revision": MODEL_REVISION,
        "dataset": DATASET_ID, "dataset_revision": DATASET_REVISION,
        "purpose": "CPU pipeline smoke test, not a quality benchmark",
        "versions": VERSIONS, "questions": QUESTIONS, "temperature": SHIPPED_T,
        "settings": {"device": DEVICE, "seed": SEED, "threads": THREADS,
                     "max_updates": MAX_UPDATES, "grad_accum": GRAD_ACCUM,
                     "micro_batch": 1, "max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN,
                     "max_debug_sequence": MAX_DEBUG_SEQUENCE, "lora_r": LORA_R,
                     "lora_alpha": LORA_ALPHA, "lora_dropout": 0.05,
                     "lora_lr": LORA_LR, "head_lr": HEAD_LR, "weight_decay": 0.01},
        "sample_ids": {"train": [r["id"] for r in train_rows], "validation": [r["id"] for r in valid_rows]},
        "checks": checks,
    }
    (OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    (OUT / "questions.json").write_text(json.dumps(QUESTIONS, indent=2), encoding="utf-8")
    (OUT / "original_decision_config.json").write_text(json.dumps(BASE_CONFIG, indent=2), encoding="utf-8")
    del head_state, model, agent
    gc.collect()
    restored = laya.load(MODEL_ID, revision=MODEL_REVISION, device="cpu", fast=False, compile=False)
    restored.model.encoder = PeftModel.from_pretrained(restored.model.encoder, OUT / "encoder_adapter", is_trainable=False)
    saved_head = load_file(str(OUT / "decision_head.safetensors"), device="cpu")
    mismatch = restored.model.load_state_dict(saved_head, strict=False)
    if mismatch.unexpected_keys or any(not k.startswith("encoder.") for k in mismatch.missing_keys):
        raise RuntimeError("The reloaded decision head architecture does not match.")
    restored.model.float().eval().requires_grad_(False)
    _, actual = evaluate_rows(restored.model, probe)
    delta = float(np.max(np.abs(expected - actual)))
    parity = {"max_probability_delta": delta, "tolerance": 1e-5, "passed": delta <= 1e-5}
    (OUT / "reload_check.json").write_text(json.dumps(parity, indent=2), encoding="utf-8")
    print(parity)
    if not parity["passed"]:
        raise RuntimeError("Predictions after reload differ beyond the tolerance. Check the package versions and weight loading.")
    print("CPU parameter updates, saving, and reload probes passed.")
    del saved_head
    gc.collect()
else:
    print("No trained weights to save.")

## 10. Download the results

After a successful run, archive the adapter, decision head, loss history, comparison, and reload check. Download the archive to keep a copy outside the Colab session.

For a GPU experiment, return to the full LoRA notebook, preserve the baseline, and set up separate splits and evaluation. The CPU check verifies the training and save/reload workflow on this runtime.

In [ ]:
if TRAINING_DONE and (OUT / "reload_check.json").exists():
    zipped = OUT.with_suffix(".zip")
    with zipfile.ZipFile(zipped, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(OUT)))
    print("Download from the Colab Files panel:", zipped.resolve())
else:
    print("Complete the CPU debug run and reload check first.")

## Four results to check

1. `lora_received_nonzero_gradient=True`: Gradients reached the adapter.
2. `lora_max_weight_change > 0`: Adapter parameters changed.
3. `frozen_weight_probe_unchanged=True`: The sampled base weights stayed unchanged, and only the expected parameters were trainable.
4. `reload_check.passed=True`: Probe predictions matched within tolerance after saving and restoring.

Also inspect loss and changes in validation predictions. Loss does not need to decrease on every update.

If one CPU update takes too long, start a new run with `MAX_UPDATES=1` and keep batch size 1. Use the full experiment for GPU performance or a larger quality evaluation.

References: [Colab resource FAQ](https://research.google.com/colaboratory/faq.html), [PEFT LoRA](https://huggingface.co/docs/peft/developer_guides/lora), [English Laya](https://huggingface.co/convaiinnovations/laya), and [Aegis dataset](https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0).

This notebook was generated without running model code locally or in Colab.